# Домашнее задание №3: Исследование KNN
**Датасет** https://www.kaggle.com/datasets/yasserh/wine-quality-dataset
 **Описание** Данный датасет содержит результаты химического анализа вин и экспертную оценку их качества; задача заключается в классификации сортов или уровней качества вина на основе его физико-химических свойств.

# Первичный анализ данных

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('WineQT.csv')

print(f"Размер датасета: {df.shape}")


print("\n--- Общая информация ---")
df.info()


print("--- Пропуски по колонкам ---")
print(df.isnull().sum())


print("\nДоля каждого класса (в %):")
print(df['quality'].value_counts(normalize=True).sort_index() * 100)

df.describe().T

Размер датасета: (1143, 13)

--- Общая информация ---
<class 'pandas.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1143 non-null   float64
 1   volatile acidity      1143 non-null   float64
 2   citric acid           1143 non-null   float64
 3   residual sugar        1143 non-null   float64
 4   chlorides             1143 non-null   float64
 5   free sulfur dioxide   1143 non-null   float64
 6   total sulfur dioxide  1143 non-null   float64
 7   density               1143 non-null   float64
 8   pH                    1143 non-null   float64
 9   sulphates             1143 non-null   float64
 10  alcohol               1143 non-null   float64
 11  quality               1143 non-null   int64  
 12  Id                    1143 non-null   int64  
dtypes: float64(11), int64(2)
memory usage: 116.2 KB
--- Пропуски по колонкам ---
fix

,count,mean,std,min,25%,50%,75%,max
fixed acidity,1143.0,8.311111,1.747595,4.60000,7.10000,7.90000,9.100000,15.90000
volatile acidity,1143.0,0.531339,0.179633,0.12000,0.39250,0.52000,0.640000,1.58000
citric acid,1143.0,0.268364,0.196686,0.00000,0.09000,0.25000,0.420000,1.00000
residual sugar,1143.0,2.532152,1.355917,0.90000,1.90000,2.20000,2.600000,15.50000
chlorides,1143.0,0.086933,0.047267,0.01200,0.07000,0.07900,0.090000,0.61100
free sulfur dioxide,1143.0,15.615486,10.250486,1.00000,7.00000,13.00000,21.000000,68.00000
total sulfur dioxide,1143.0,45.914698,32.782130,6.00000,21.00000,37.00000,61.000000,289.00000
density,1143.0,0.996730,0.001925,0.99007,0.99557,0.99668,0.997845,1.00369
pH,1143.0,3.311015,0.156664,2.74000,3.20500,3.31000,3.400000,4.01000
sulphates,1143.0,0.657708,0.170399,0.33000,0.55000,0.62000,0.730000,2.00000


# Подготовка данных

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#Пропусков в моем датасете нет

#Так как все колонки числовые(float либо int) то кодирование мне не нужно

#Разделим выборку на train и test
X = df.drop(['quality', 'Id'], axis=1)
y = df['quality']
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print(f"Обучающая выборка (X_train): {X_train.shape[0]} строк")
print(f"Тестовая выборка (X_test): {X_test.shape[0]} строк")

#Далее проведем масштабирование
scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)




Обучающая выборка (X_train): 800 строк
Тестовая выборка (X_test): 343 строк


**почему масштабирование важно для KNN** Потому что как мы узнали KNN это метрический метод, соответсвенно если один категориальный признак измеряется в единицах, а другой в сотнях, то признак с большими числами будет доминировать, так как важен квадрат расстояния. **почему нельзя подбирать параметры на тестовой выборке** Во первых тестовая выборка предназначена исключительно для финальной оценки модели, во вторых таким образом можно подобрать иделальные параметры под эти 30 процентов выборки и придти к переобучению.

# Обучение KNN

In [14]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

weights = ['uniform', 'distance']
metrics = ['euclidean', 'manhattan', 'minkowski']

results = []
for m in metrics:
    for w in weights:
        for k in range(1,30):
            knn = KNeighborsClassifier(n_neighbors=k, metric=m,weights=w)
            knn.fit(X_train_scaled, y_train)
            y_pred = knn.predict(X_test_scaled)
            acc = accuracy_score(y_test, y_pred)

            results.append({
            'metric': m,
            'accuracy': acc,
            'weight': w,
            'число соседей ': k
             })

results_df = pd.DataFrame(results)
# Снимаем ограничения на количество строк и столбцов
pd.set_option('display.max_rows', None)    # Показывать все строки
pd.set_option('display.max_columns', None) # Показывать все столбцы
pd.set_option('display.width', 1000)       # Чтобы таблица не переносилась на новую строку

# Теперь выводим
print(results_df)


        metric  accuracy    weight  число соседей 
0    euclidean  0.615160   uniform               1
1    euclidean  0.574344   uniform               2
2    euclidean  0.597668   uniform               3
3    euclidean  0.600583   uniform               4
4    euclidean  0.586006   uniform               5
5    euclidean  0.594752   uniform               6
6    euclidean  0.594752   uniform               7
7    euclidean  0.597668   uniform               8
8    euclidean  0.612245   uniform               9
9    euclidean  0.603499   uniform              10
10   euclidean  0.603499   uniform              11
11   euclidean  0.603499   uniform              12
12   euclidean  0.580175   uniform              13
13   euclidean  0.603499   uniform              14
14   euclidean  0.568513   uniform              15
15   euclidean  0.580175   uniform              16
16   euclidean  0.586006   uniform              17
17   euclidean  0.577259   uniform              18
18   euclidean  0.568513   unif

**Про влияние типа весов** Можно заметить, что когда мы раздаем веса пропорциально расстоянию до обьекта мы спокойно доходим 0.66-0.67 точности, если же не делаем этого то доходим до 0.59-0.60, то есть веса дейтсвительно очень много решают. **Число соседей** Для uniform можно увидеть что не особо влияет, точность скачет около примерно одного уровня, когда же для distance до определенного момента точность растет, примерно до 14.**Метрики** показывают примерно одинаковые результаты.

# Подбор гиперпарметров

In [15]:
from sklearn.model_selection import  GridSearchCV

param_grid = {
    'n_neighbors': range(1, 31),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

grid_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучшая точность на кросс-валидации: {grid_search.best_score_:.4f}")



C:\Users\viteb\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


Лучшие параметры: {'metric': 'euclidean', 'n_neighbors': 29, 'weights': 'distance'}
Лучшая точность на кросс-валидации: 0.6275


# Выводы
В этой работе я узнал про реализацию метрического метода KNN. Проверил влияние различного типа весов, метрик, числа соседей и сравнил результаты. Попробовал применять масштабирование и кросс-валидацию. С помощью этого нашел гиперпарметры для моего датасета.